In [17]:
import os, random
import cv2 as cv
from pathlib import Path

In [62]:
### variabels for input and output path
input_path = './data/pre/'
output_path = './data/post/'

### dict with class labels
class_lables = {
    "Brown bear": 0,
    "Canary": 1,
    "Cheetah": 2,
    "Crocodile": 3,
    "Elephant": 4,
    "Fox": 5,
    "Goat": 6,
    "Goldfish": 7,
    "Kangaroo": 8,
    "Leopard": 9,
    "Mouse": 10,
    "Ostrich": 11,
    "Panda": 12,
    "Pig": 13,
    "Rabbit": 14,
    "Raccoon": 15,
    "Rhinoceros": 16,
    "Sheep": 17,
    "Woodpecker": 18,
    "Zebra": 19,
}

In [66]:
#### Helper function for transforming label file to correct format
def transform_label_file(input_path, output_path, width, height):
    ### creating output variabel
    output = ""
    ### reading input file
    with open(input_path, 'r') as f:
        content = f.read().split()
        ### transforming the values in the label file
        x_center = round(((float(content[1]) + float(content[3])) / 2 ) / width, 4)
        y_center = round(((float(content[2]) + float(content[4])) / 2 ) / height, 4)
        image_width = round((float(content[3]) - float(content[1])) / width,4)
        image_height = round((float(content[4]) - float(content[2])) / height,4)
        ### creating the output for the new transformed label file
        output = str(class_lables.get(content[0])) + " " + str(x_center) + " " + str(y_center) + " " + str(image_width) + " " + str(image_height)
    
    ### writing the output to the new label file
    with open(output_path, "w") as f:
        f.write(output);

def pre_process_images(input_path, output_path):       
    for root, dirs, file in os.walk(input_path):
        for d in dirs:
            if d == "Pig" or d == "Sheep" or d == "Zebra":
                image_paths = []
                d_path = os.path.join(root, d)
                print(d_path)
                for file in os.listdir(d_path):
                    if file.endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        image_paths.append(file)
                random.shuffle(image_paths)
                count = 0
                split = int(len(image_paths)*0.15)
                for file in image_paths:
                    path = os.path.join(d_path, file)
                    img = cv.imread(path)
                    label_input_path = d_path + "/Label/" + Path(file).stem + ".txt"                 
                    file_path = d + str(count) + Path(path).suffix
                    label_path = d + str(count) + ".txt"
                    if count < split:
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/test/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/test/" + file_path), img)
                    elif count >= split and count < split*2:
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/val/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/val/" + file_path), img)
                    else:
                        transform_label_file(label_input_path, os.path.join(output_path, "labels/train/" + label_path), img.shape[1], img.shape[0])
                        cv.imwrite(os.path.join(output_path, "images/train/" + file_path), img)
                    count += 1


In [11]:
transform_label_file('./data/pre/Pig/Label/0a1e3ea3cf2990b8.txt', "./data/post/labels/train/test.txt", 1024, 768)

In [67]:
pre_process_images(input_path, output_path)

./data/pre/Pig
./data/pre/Sheep
./data/pre/Zebra
